<a href="https://colab.research.google.com/github/Mohd-Dilshan/ai-voice-sentiment-analyzer/blob/main/audio_sentiment_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install openai-whisper
!apt install ffmpeg

In [ ]:
# Import libraries
import whisper
from google.colab import files
from transformers import pipeline

# Load Whisper model
model = whisper.load_model("base")

# Upload audio file
uploaded = files.upload()

# Get file name
audio = list(uploaded.keys())[0]

# Transcribe audio
result = model.transcribe(audio)
transcribe = result["text"]

print("Transcription:")
print(transcribe)

# Sentiment analysis
sentiment = pipeline("sentiment-analysis")

sentiment_result = sentiment(transcribe)

print("Sentiment Result:")
print(sentiment_result)

In [ ]:
!pip install flask transformers pyngrok

In [ ]:
!mkdir templates

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>
<head>
<title>Audio Sentiment Analyzer</title>
</head>

<body>

<h2>Upload Audio File</h2>

<form action="/upload" method="post" enctype="multipart/form-data">

<input type="file" name="audio" required>

<br><br>

<button type="submit">Analyze Audio</button>

</form>

</body>
</html>

In [ ]:
%%writefile templates/result.html
<!DOCTYPE html>
<html>
<head>
<title>Result</title>
</head>

<body>

<h2>Transcription</h2>
<p>{{ transcription }}</p>

<h2>Sentiment</h2>
<p>{{ sentiment }}</p>

<br>

<a href="/">Upload Another Audio</a>

</body>
</html>

In [ ]:
%%writefile app.py

import whisper
from flask import Flask, request, render_template
from transformers import pipeline
import os

# Initialize Flask app
app = Flask(__name__)

# Load models
model = whisper.load_model("base")
sentiment = pipeline("sentiment-analysis")

# Home page
@app.route("/")
def home():
    return render_template("index.html")

# Upload route
@app.route("/upload", methods=["POST"])
def upload():

    file = request.files["audio"]

    filename = file.filename
    filepath = os.path.join(filename)

    file.save(filepath)

    # Transcribe audio
    result = model.transcribe(filepath)
    text = result["text"]

    # Sentiment analysis
    sentiment_result = sentiment(text)

    return render_template(
        "result.html",
        transcription=text,
        sentiment=sentiment_result
    )

if __name__ == "__main__":
    app.run()

In [ ]:
from pyngrok import ngrok
import subprocess

In [ ]:
!pip install gradio

In [ ]:
import whisper
from transformers import pipeline
import gradio as gr

# Load models
model = whisper.load_model("base")
sentiment = pipeline("sentiment-analysis")

# Function to process audio
def analyze_audio(audio):

    result = model.transcribe(audio)
    text = result["text"]

    sentiment_result = sentiment(text)

    return text, sentiment_result

# Create web interface
app = gr.Interface(
    fn=analyze_audio,
    inputs=gr.Audio(type="filepath"),
    outputs=["text","text"],
    title="Audio Sentiment Analyzer"
)

# Launch web app
app.launch(share=True)